In [62]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

try:
    current_path = Path(__file__).resolve()
except NameError:
    current_path = Path().resolve()

for parent in current_path.parents:
    if (parent / "stock_forecast" / "DATA").is_dir():
        stock_forecast_path = parent / "stock_forecast"
        break
else:
    raise ImportError("stock_forecast/DATA 폴더를 찾을 수 없습니다.")

if str(stock_forecast_path) not in sys.path:
    sys.path.insert(0, str(stock_forecast_path))

# from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler, RobustScaler
from DATA.stock_invest_function import *
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta

def clean_numeric_data(series, method='drop'):
    series = series.replace([np.inf, -np.inf], np.nan)
    if method == 'drop':
        return series.dropna()
    elif method == 'fill_median':
        return series.fillna(series.median())
    elif method == 'fill_mean':
        return series.fillna(series.mean())
    elif method == 'fill_zero':
        return series.fillna(0)
    else:
        return series

def get_market_cap_by_ticker(db_info: dict, ticker: str) -> pd.DataFrame:
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )

        query = f"""
        SELECT date, value FROM ks_listed_company_daily_marketcap
        WHERE ticker = '{ticker}' AND indicator = '시가총액'
        ORDER BY date
        """

        df = pd.read_sql(query, con=engine)
        return df

    except Exception as e:
        print(f"시가총액 데이터 조회 실패: {e}")
        return pd.DataFrame()

def save_valuation_to_db(db_info: dict, table_name: str, df: pd.DataFrame):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )

        new_ticker = df['ticker'].iloc[0]

        # 기존 데이터 삭제
        delete_query = f"DELETE FROM {table_name} WHERE ticker = '{new_ticker}'"

        with engine.connect() as conn:
            conn.execute(text(delete_query))
            conn.commit()

        # 새 데이터 추가
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists='append',
            index=False,
            method='multi'
        )

    except Exception as e:
        print(f"DB 저장 실패: {e}")

# 사용자 설정
tic_name = 'A042700 '
hs_code = None  # None으로 설정 시 외생변수 없이 예측
st_date = '2010-01-01'
end_date = '2025-10-31'
today_date = pd.to_datetime(datetime.today().date())

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

# 1단계: 매출 데이터 추출
fs_df = fetch_table_data(db_info, "korea_fs_data")
fs_df.rename(columns={'Date': 'date'}, inplace=True)

target_indicator = '매출액(천원)'
revenue_raw = fs_df[fs_df['indicator'] == target_indicator].copy()
revenue_company = revenue_raw[revenue_raw['symbol'] == tic_name].copy()

if len(revenue_company) == 0:
    print(f"해당 기업의 매출 데이터가 없습니다.")
    exit()

revenue_company['date'] = pd.to_datetime(revenue_company['date'])
revenue_company['value'] = pd.to_numeric(revenue_company['value'], errors='coerce')
revenue_company = revenue_company.dropna(subset=['value']).sort_values('date')

revenue_company['year'] = revenue_company['date'].dt.year
revenue_company['quarter'] = revenue_company['date'].dt.quarter
revenue_company['year_quarter'] = revenue_company['year'].astype(str) + 'Q' + revenue_company['quarter'].astype(str)

revenue_quarterly = revenue_company.groupby(['year', 'quarter']).agg({
    'date': 'last',
    'value': 'last',
    'year_quarter': 'last',
    'symbol': 'last'
}).reset_index()

revenue_quarterly = revenue_quarterly.sort_values(['year', 'quarter']).reset_index(drop=True)
revenue_quarterly['revenue'] = clean_numeric_data(revenue_quarterly['value'], method='fill_median')
revenue_data_extracted = revenue_quarterly.copy()

# 2단계: 수출 데이터 추출 (hs_code가 None이 아닌 경우만)
if hs_code is not None:
    export_df = fetch_table_data(db_info, "korea_monthly_trade_data_forecast")
    export_company = export_df[export_df['root_hs_code'] == hs_code].copy()

    if len(export_company) > 0:
        export_company['date'] = pd.to_datetime(export_company['date'])
        export_company['expDlr_forecast_12m'] = pd.to_numeric(export_company['expDlr_forecast_12m'], errors='coerce')
        export_company = export_company.dropna(subset=['expDlr_forecast_12m']).sort_values('date')

        export_company['year'] = export_company['date'].dt.year
        export_company['quarter'] = export_company['date'].dt.quarter
        export_company['year_quarter'] = export_company['year'].astype(str) + 'Q' + export_company['quarter'].astype(str)
        export_company['month'] = export_company['date'].dt.month

        quarter_end_months = {1: 3, 2: 6, 3: 9, 4: 12}
        quarter_check = export_company.groupby(['year', 'quarter']).agg({
            'month': 'max',
            'date': 'count'
        }).reset_index()

        complete_quarters = []
        for _, row in quarter_check.iterrows():
            expected_end_month = quarter_end_months[row['quarter']]
            if row['month'] == expected_end_month:
                complete_quarters.append((row['year'], row['quarter']))

        complete_quarter_filter = export_company.apply(
            lambda x: (x['year'], x['quarter']) in complete_quarters, axis=1
        )
        export_company_filtered = export_company[complete_quarter_filter].copy()

        export_quarterly = export_company_filtered.groupby(['year', 'quarter']).agg({
            'expDlr_forecast_12m': 'sum',
            'date': 'last',
            'year_quarter': 'last',
            'root_hs_code': 'last'
        }).reset_index()

        export_quarterly = export_quarterly.sort_values(['year', 'quarter']).reset_index(drop=True)
        export_data_extracted = export_quarterly.copy()
    else:
        export_data_extracted = pd.DataFrame()
else:
    export_data_extracted = pd.DataFrame()

# 3단계: YoY 성장률 계산 및 SARIMA 예측
if hs_code is not None and len(export_data_extracted) > 0:
    export_yoy_df = export_data_extracted.copy()
    export_yoy_df['exog_var'] = np.nan
    for i in range(4, len(export_yoy_df)):
        if export_yoy_df.iloc[i-4]['expDlr_forecast_12m'] != 0:
            yoy_rate = (export_yoy_df.iloc[i]['expDlr_forecast_12m'] / export_yoy_df.iloc[i-4]['expDlr_forecast_12m'] - 1) * 100
            export_yoy_df.loc[export_yoy_df.index[i], 'exog_var'] = yoy_rate

    export_exog_df = export_yoy_df[['date', 'exog_var']].dropna().copy()
else:
    export_exog_df = pd.DataFrame()

revenue_endog_df = revenue_data_extracted[['date', 'revenue']].copy()
revenue_endog_df.rename(columns={'revenue': 'endog_var'}, inplace=True)

if len(export_exog_df) > 0:
    combined_df = pd.merge(revenue_endog_df, export_exog_df, on='date', how='outer').sort_values('date')
else:
    combined_df = revenue_endog_df.copy()
    combined_df['exog_var'] = np.nan

final_combined_data = combined_df.copy()
forecast_df = combined_df[combined_df['endog_var'].notna()].copy()

if len(forecast_df) >= 8:
    try:
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        from itertools import product

        endog = forecast_df['endog_var'].values
        exog = forecast_df['exog_var'].values if (hs_code is not None and forecast_df['exog_var'].notna().any()) else None

        if exog is not None:
            exog_series = pd.Series(exog)
            exog_series = exog_series.interpolate(method='linear').fillna(method='ffill').fillna(method='bfill')
            exog = exog_series.values

        p_values = [0, 1, 2]
        d_values = [0, 1]
        q_values = [0, 1, 2]
        P_values = [0, 1]
        D_values = [0, 1]
        Q_values = [0, 1]
        s_value = 4

        best_aic = float('inf')
        best_params = None
        best_model = None

        for p, d, q, P, D, Q in product(p_values, d_values, q_values, P_values, D_values, Q_values):
            try:
                total_params = p + q + P + Q + 1
                if total_params >= len(endog) * 0.3:
                    continue

                model = SARIMAX(
                    endog,
                    exog=exog,
                    order=(p, d, q),
                    seasonal_order=(P, D, Q, s_value),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                fitted_model = model.fit(disp=False, maxiter=100)

                if np.isfinite(fitted_model.aic) and fitted_model.aic < best_aic:
                    best_aic = fitted_model.aic
                    best_params = (p, d, q, P, D, Q, s_value)
                    best_model = fitted_model

            except Exception:
                continue

        if best_model is None:
            model = SARIMAX(endog, exog=exog, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0))
            best_model = model.fit(disp=False)
            best_params = (1, 1, 1, 0, 0, 0, 0)

        if exog is not None:
            recent_exog = forecast_df['exog_var'].tail(4).mean()
            future_exog = [recent_exog] * 4
            forecast_result = best_model.forecast(steps=4, exog=future_exog)
        else:
            forecast_result = best_model.forecast(steps=4)

        # 외생변수 없는 모델
        endog_no_exog = forecast_df['endog_var'].values
        use_log_transform = False
        if np.all(endog_no_exog > 0):
            try:
                endog_log = np.log(endog_no_exog)
                use_log_transform = True
                endog_for_model = endog_log
            except:
                endog_for_model = endog_no_exog
        else:
            endog_for_model = endog_no_exog

        simple_params = [
            (0, 1, 0, 0, 0, 0, 0),
            (1, 1, 0, 0, 0, 0, 0),
            (0, 1, 1, 0, 0, 0, 0),
            (1, 1, 1, 0, 0, 0, 0),
            (1, 0, 0, 0, 0, 0, 0),
            (0, 0, 1, 0, 0, 0, 0),
        ]

        best_aic_no_exog = float('inf')
        best_model_no_exog = None

        for params in simple_params:
            p, d, q, P, D, Q, s = params
            try:
                model_no_exog = SARIMAX(
                    endog_for_model,
                    order=(p, d, q),
                    seasonal_order=(P, D, Q, s),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                fitted_model_no_exog = model_no_exog.fit(disp=False, maxiter=50)

                if np.isfinite(fitted_model_no_exog.aic) and fitted_model_no_exog.aic < best_aic_no_exog:
                    best_aic_no_exog = fitted_model_no_exog.aic
                    best_model_no_exog = fitted_model_no_exog

            except Exception:
                continue

        if best_model_no_exog is None:
            try:
                model_no_exog = SARIMAX(endog_for_model, order=(0, 1, 0))
                best_model_no_exog = model_no_exog.fit(disp=False, maxiter=30)
            except:
                best_model_no_exog = None

        forecast_result_no_exog = None
        if best_model_no_exog is not None:
            try:
                forecast_result_no_exog = best_model_no_exog.forecast(steps=4)
                if use_log_transform:
                    forecast_result_no_exog = np.exp(forecast_result_no_exog)
            except Exception:
                best_model_no_exog = None

        if best_model_no_exog is None or forecast_result_no_exog is None:
            recent_revenues = endog_no_exog[-8:]
            if len(recent_revenues) >= 4:
                yoy_growth_rates = []
                for i in range(4, len(recent_revenues)):
                    if recent_revenues[i-4] != 0:
                        growth = (recent_revenues[i] / recent_revenues[i-4] - 1)
                        yoy_growth_rates.append(growth)

                if yoy_growth_rates:
                    avg_growth = np.mean(yoy_growth_rates)
                    conservative_growth = avg_growth * 0.8
                else:
                    conservative_growth = 0.02
            else:
                conservative_growth = 0.02

            last_revenue = endog_no_exog[-1]
            forecast_result_no_exog = []
            for i in range(4):
                pred_value = last_revenue * (1 + conservative_growth) ** (i + 1)
                forecast_result_no_exog.append(pred_value)

            forecast_result_no_exog = np.array(forecast_result_no_exog)

        last_date = forecast_df.iloc[-1]['date']
        future_dates = []
        for i in range(1, 5):
            future_date = last_date + pd.DateOffset(months=3*i)
            future_date = future_date + pd.offsets.QuarterEnd(0)
            future_dates.append(future_date)

        forecast_results_df = pd.DataFrame({
            'date': future_dates,
            'predicted_revenue': forecast_result,
            'predicted_revenue_without_exog': forecast_result_no_exog
        })

        forecast_results_df['year'] = forecast_results_df['date'].dt.year
        forecast_results_df['quarter'] = forecast_results_df['date'].dt.quarter
        forecast_results_df['year_quarter'] = (forecast_results_df['year'].astype(str) + 'Q' +
                                               forecast_results_df['quarter'].astype(str))

        sarima_forecast_results = forecast_results_df.copy()

    except Exception as e:
        print(f"SARIMA 예측 실패: {str(e)}")
        sarima_forecast_results = pd.DataFrame()

# 4단계: 실제+예측 매출 결합 및 TTM 계산
actual_revenue_df = revenue_data_extracted[['date', 'revenue']].copy()
actual_revenue_df['date'] = pd.to_datetime(actual_revenue_df['date'])
actual_revenue_df = actual_revenue_df.sort_values('date').reset_index(drop=True)

if len(sarima_forecast_results) > 0:
    forecast_revenue_df = sarima_forecast_results[['date', 'predicted_revenue', 'predicted_revenue_without_exog']].copy()
    forecast_revenue_df['date'] = pd.to_datetime(forecast_revenue_df['date'])
    forecast_revenue_df = forecast_revenue_df.sort_values('date').reset_index(drop=True)
else:
    forecast_revenue_df = pd.DataFrame()

actual_revenue_df['predicted_revenue'] = np.nan
actual_revenue_df['predicted_revenue_without_exog'] = np.nan

if len(forecast_revenue_df) > 0:
    forecast_revenue_df['revenue'] = np.nan

    actual_revenue_df = actual_revenue_df[['date', 'revenue', 'predicted_revenue', 'predicted_revenue_without_exog']]
    forecast_revenue_df = forecast_revenue_df[['date', 'revenue', 'predicted_revenue', 'predicted_revenue_without_exog']]

    combined_revenue_df = pd.concat([actual_revenue_df, forecast_revenue_df], ignore_index=True)
else:
    combined_revenue_df = actual_revenue_df.copy()

combined_revenue_df = combined_revenue_df.sort_values('date').reset_index(drop=True)

combined_revenue_df['revenue_with_exog_forecast'] = combined_revenue_df['revenue'].fillna(combined_revenue_df['predicted_revenue'])
combined_revenue_df['revenue_without_exog_forecast'] = combined_revenue_df['revenue'].fillna(combined_revenue_df['predicted_revenue_without_exog'])

def calculate_ttm(series):
    return series.rolling(window=4, min_periods=1).sum()

combined_revenue_df['ttm_revenue_with_exog'] = calculate_ttm(combined_revenue_df['revenue_with_exog_forecast'])
combined_revenue_df['ttm_revenue_without_exog'] = calculate_ttm(combined_revenue_df['revenue_without_exog_forecast'])
combined_revenue_df['ttm_actual_revenue'] = calculate_ttm(combined_revenue_df['revenue'])

combined_revenue_df['year'] = combined_revenue_df['date'].dt.year
combined_revenue_df['quarter'] = combined_revenue_df['date'].dt.quarter
combined_revenue_df['year_quarter'] = (combined_revenue_df['year'].astype(str) + 'Q' +
                                      combined_revenue_df['quarter'].astype(str))

last_actual_date = actual_revenue_df['date'].max()
combined_revenue_df['is_forecast'] = combined_revenue_df['date'] > last_actual_date

ttm_combined_data = combined_revenue_df.copy()

# 5단계: 월별 PSR 추정 및 향후 12개월 PSR 예측
market_cap_df = get_market_cap_by_ticker(db_info, tic_name)

if len(market_cap_df) == 0:
    print(f"{tic_name} 기업의 시가총액 데이터가 없습니다.")
    exit()

market_cap_df['date'] = pd.to_datetime(market_cap_df['date'])
market_cap_df['market_cap'] = pd.to_numeric(market_cap_df['value'], errors='coerce')
market_cap_df = market_cap_df.dropna(subset=['market_cap']).sort_values('date')

market_cap_monthly = market_cap_df.set_index('date')['market_cap'].resample('M').last().reset_index()
market_cap_monthly = market_cap_monthly.dropna()

full_date_range = pd.date_range(
    start=market_cap_monthly['date'].min(),
    end=market_cap_monthly['date'].max(),
    freq='M'
)

market_cap_complete = market_cap_monthly.set_index('date').reindex(full_date_range)
market_cap_complete.index.name = 'date'
market_cap_complete['market_cap'] = market_cap_complete['market_cap'].interpolate(method='linear')
market_cap_complete['market_cap'] = market_cap_complete['market_cap'].fillna(method='ffill').fillna(method='bfill')
market_cap_monthly = market_cap_complete.reset_index()

actual_revenue_ttm = ttm_combined_data[ttm_combined_data['is_forecast'] == False].copy()
actual_revenue_ttm = actual_revenue_ttm[['date', 'revenue']].copy()
actual_revenue_ttm['date'] = pd.to_datetime(actual_revenue_ttm['date'])
actual_revenue_ttm = actual_revenue_ttm.sort_values('date').reset_index(drop=True)
actual_revenue_ttm["ttm_revenue"] = actual_revenue_ttm["revenue"].rolling(window=4).sum()
psr_calculation_df = pd.merge(market_cap_monthly, actual_revenue_ttm, on="date", how="left").ffill(limit=4).dropna()

psr_calculation_df['psr'] = psr_calculation_df['market_cap'] / ((psr_calculation_df['ttm_revenue'] + 1e-10)*1000)

psr_calculation_df = psr_calculation_df[
    (psr_calculation_df['psr'] > 0) &
    (psr_calculation_df['psr'] < psr_calculation_df['psr'].quantile(0.99))
]

if len(psr_calculation_df) >= 60:
    try:
        from statsmodels.tsa.statespace.sarimax import SARIMAX

        psr_series = psr_calculation_df['psr'].values

        use_log_transform_psr = False
        if np.all(psr_series > 0):
            try:
                psr_log = np.log(psr_series)
                use_log_transform_psr = True
                psr_for_model = psr_log
            except:
                psr_for_model = psr_series
        else:
            psr_for_model = psr_series

        psr_params = [
            (0, 1, 0, 0, 0, 0, 0),
            (1, 1, 0, 0, 0, 0, 0),
            (0, 1, 1, 0, 0, 0, 0),
            (1, 1, 1, 0, 0, 0, 0),
            (1, 0, 0, 0, 0, 0, 0),
            (0, 0, 1, 0, 0, 0, 0),
            (1, 1, 1, 1, 0, 1, 12),
            (0, 1, 1, 0, 1, 1, 12),
        ]

        best_aic_psr = float('inf')
        best_model_psr = None

        for params in psr_params:
            p, d, q, P, D, Q, s = params
            try:
                model_psr = SARIMAX(
                    psr_for_model,
                    order=(p, d, q),
                    seasonal_order=(P, D, Q, s),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                fitted_model_psr = model_psr.fit(disp=False, maxiter=100)

                if np.isfinite(fitted_model_psr.aic) and fitted_model_psr.aic < best_aic_psr:
                    best_aic_psr = fitted_model_psr.aic
                    best_model_psr = fitted_model_psr

            except Exception:
                continue

        if best_model_psr is None:
            try:
                model_psr = SARIMAX(psr_for_model, order=(0, 1, 0))
                best_model_psr = model_psr.fit(disp=False, maxiter=30)
            except:
                best_model_psr = None

        psr_forecast = None
        if best_model_psr is not None:
            try:
                psr_forecast = best_model_psr.forecast(steps=12)
                if use_log_transform_psr:
                    psr_forecast = np.exp(psr_forecast)
            except Exception:
                best_model_psr = None

        if best_model_psr is None or psr_forecast is None:
            recent_psr = psr_series[-12:]
            avg_psr = np.mean(recent_psr)
            psr_forecast = np.array([avg_psr] * 12)

        last_actual_date = psr_calculation_df['date'].max()
        future_dates_psr = pd.date_range(
            start=last_actual_date + pd.DateOffset(months=1),
            periods=12,
            freq='M'
        )

        psr_forecast_df = pd.DataFrame({
            'date': future_dates_psr,
            'predicted_psr': psr_forecast
        })

        psr_forecast_df['year'] = psr_forecast_df['date'].dt.year
        psr_forecast_df['month'] = psr_forecast_df['date'].dt.month
        psr_forecast_df['year_month'] = psr_forecast_df['date'].dt.strftime('%Y-%m')

        psr_calculation_df['predicted_psr'] = np.nan
        psr_calculation_df['is_forecast'] = False

        psr_forecast_df['psr'] = np.nan
        psr_forecast_df['market_cap'] = np.nan
        psr_forecast_df['ttm_revenue'] = np.nan
        psr_forecast_df['is_forecast'] = True

        common_cols = ['date', 'psr', 'predicted_psr', 'is_forecast']

        psr_combined = pd.concat([
            psr_calculation_df[common_cols],
            psr_forecast_df[common_cols]
        ], ignore_index=True)

        psr_combined = psr_combined.sort_values('date').reset_index(drop=True)
        psr_combined['combined_psr'] = psr_combined['psr'].fillna(psr_combined['predicted_psr'])

        monthly_psr_data = psr_calculation_df.copy()
        monthly_psr_forecast = psr_forecast_df.copy()
        monthly_psr_combined = psr_combined.copy()

    except Exception:
        monthly_psr_forecast = pd.DataFrame()
else:
    monthly_psr_forecast = pd.DataFrame()

# 6단계: LSTM과 Prophet을 이용한 12개월 PSR 예측
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.optimizers import Adam
    from sklearn.preprocessing import MinMaxScaler
    lstm_available = True
except ImportError:
    lstm_available = False

try:
    from prophet import Prophet
    prophet_available = True
except ImportError:
    prophet_available = False

if 'monthly_psr_data' in locals():
    psr_data = monthly_psr_data[['date', 'psr']].copy()
    psr_data = psr_data.sort_values('date').reset_index(drop=True)

    lstm_forecast = None
    if lstm_available and len(psr_data) >= 24:
        try:
            psr_values = psr_data['psr'].values.reshape(-1, 1)
            scaler = MinMaxScaler(feature_range=(0, 1))
            psr_scaled = scaler.fit_transform(psr_values)

            def create_sequences(data, seq_length):
                X, y = [], []
                for i in range(seq_length, len(data)):
                    X.append(data[i-seq_length:i, 0])
                    y.append(data[i, 0])
                return np.array(X), np.array(y)

            seq_length = min(12, len(psr_scaled) // 3)

            if len(psr_scaled) > seq_length:
                X, y = create_sequences(psr_scaled, seq_length)

                train_size = int(len(X) * 0.8)
                X_train, X_val = X[:train_size], X[train_size:]
                y_train, y_val = y[:train_size], y[train_size:]

                X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
                X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))

                model = Sequential([
                    LSTM(50, return_sequences=True, input_shape=(seq_length, 1)),
                    Dropout(0.2),
                    LSTM(50, return_sequences=False),
                    Dropout(0.2),
                    Dense(25),
                    Dense(1)
                ])

                model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
                tf.random.set_seed(42)
                history = model.fit(
                    X_train, y_train,
                    batch_size=32,
                    epochs=50,
                    validation_data=(X_val, y_val),
                    verbose=0
                )

                last_sequence = psr_scaled[-seq_length:].reshape(1, seq_length, 1)
                lstm_predictions = []

                for _ in range(12):
                    next_pred = model.predict(last_sequence, verbose=0)
                    lstm_predictions.append(next_pred[0, 0])
                    last_sequence = np.roll(last_sequence, -1, axis=1)
                    last_sequence[0, -1, 0] = next_pred[0, 0]

                lstm_predictions = np.array(lstm_predictions).reshape(-1, 1)
                lstm_forecast = scaler.inverse_transform(lstm_predictions).flatten()

        except Exception:
            lstm_forecast = None

    prophet_forecast = None
    if prophet_available and len(psr_data) >= 24:
        try:
            prophet_data = psr_data.copy()
            prophet_data.columns = ['ds', 'y']

            model_prophet = Prophet(
                daily_seasonality=False,
                weekly_seasonality=False,
                yearly_seasonality=True,
                seasonality_mode='multiplicative',
                changepoint_prior_scale=0.1,
                seasonality_prior_scale=0.1
            )

            model_prophet.fit(prophet_data)
            future_dates = model_prophet.make_future_dataframe(periods=12, freq='M')
            forecast_prophet = model_prophet.predict(future_dates)
            prophet_forecast = forecast_prophet.tail(12)['yhat'].values
            prophet_forecast = np.maximum(prophet_forecast, 0.01)

        except Exception:
            prophet_forecast = None

    last_date = psr_data['date'].max()
    future_dates = pd.date_range(
        start=last_date + pd.DateOffset(months=1),
        periods=12,
        freq='M'
    )

    forecast_comparison_df = pd.DataFrame({
        'date': future_dates,
        'year_month': future_dates.strftime('%Y-%m')
    })

    if 'monthly_psr_forecast' in locals() and len(monthly_psr_forecast) > 0:
        sarima_forecast = monthly_psr_forecast['predicted_psr'].values
        forecast_comparison_df['sarima_forecast'] = sarima_forecast
    else:
        forecast_comparison_df['sarima_forecast'] = np.nan

    if lstm_forecast is not None:
        forecast_comparison_df['lstm_forecast'] = lstm_forecast
    else:
        forecast_comparison_df['lstm_forecast'] = np.nan

    if prophet_forecast is not None:
        forecast_comparison_df['prophet_forecast'] = prophet_forecast
    else:
        forecast_comparison_df['prophet_forecast'] = np.nan

    valid_forecasts = []

    if not pd.isna(forecast_comparison_df['sarima_forecast']).all():
        valid_forecasts.append(forecast_comparison_df['sarima_forecast'])

    if not pd.isna(forecast_comparison_df['lstm_forecast']).all():
        valid_forecasts.append(forecast_comparison_df['lstm_forecast'])

    if not pd.isna(forecast_comparison_df['prophet_forecast']).all():
        valid_forecasts.append(forecast_comparison_df['prophet_forecast'])

    if valid_forecasts:
        ensemble_forecast = np.mean(valid_forecasts, axis=0)
        forecast_comparison_df['ensemble_forecast'] = ensemble_forecast
    else:
        forecast_comparison_df['ensemble_forecast'] = np.nan

    psr_forecast_comparison = forecast_comparison_df.copy()

else:
    psr_forecast_comparison = pd.DataFrame()

# 7단계: PSR과 TTM 매출을 이용한 가치 추정
if len(psr_forecast_comparison) > 0:
    ttm_revenue_data = ttm_combined_data[['date', 'ttm_revenue_with_exog', 'ttm_revenue_without_exog']].copy()
    ttm_revenue_data['date'] = pd.to_datetime(ttm_revenue_data['date'])
    ttm_revenue_data = ttm_revenue_data.sort_values('date').reset_index(drop=True)

    ttm_revenue_data['year'] = ttm_revenue_data['date'].dt.year
    ttm_revenue_data['quarter'] = ttm_revenue_data['date'].dt.quarter

    valuation_df = psr_forecast_comparison.copy()
    valuation_df['date'] = pd.to_datetime(valuation_df['date'])

    valuation_df['year'] = valuation_df['date'].dt.year
    valuation_df['month'] = valuation_df['date'].dt.month
    valuation_df['quarter'] = valuation_df['date'].dt.quarter

    def get_ttm_revenue_for_quarter(forecast_year, forecast_quarter, ttm_data):
        if forecast_quarter == 1:
            target_year = forecast_year - 1
            target_quarter = 4
        elif forecast_quarter == 2:
            target_year = forecast_year
            target_quarter = 1
        elif forecast_quarter == 3:
            target_year = forecast_year
            target_quarter = 2
        elif forecast_quarter == 4:
            target_year = forecast_year
            target_quarter = 3
        else:
            return None, None

        target_data = ttm_data[
            (ttm_data['year'] == target_year) &
            (ttm_data['quarter'] == target_quarter)
        ]

        if len(target_data) > 0:
            target_row = target_data.iloc[-1]
            return target_row['ttm_revenue_with_exog'], target_row['ttm_revenue_without_exog']
        else:
            return None, None

    valuation_df['matched_ttm_with_exog'] = np.nan
    valuation_df['matched_ttm_without_exog'] = np.nan

    for i, row in valuation_df.iterrows():
        forecast_year = row['year']
        forecast_quarter = row['quarter']

        ttm_with_exog, ttm_without_exog = get_ttm_revenue_for_quarter(
            forecast_year, forecast_quarter, ttm_revenue_data
        )

        if ttm_with_exog is not None:
            valuation_df.loc[i, 'matched_ttm_with_exog'] = ttm_with_exog
            valuation_df.loc[i, 'matched_ttm_without_exog'] = ttm_without_exog

    psr_columns = ['sarima_forecast', 'lstm_forecast', 'prophet_forecast', 'ensemble_forecast']

    for psr_col in psr_columns:
        valuation_col = psr_col.replace('_forecast', '_valuation')
        valuation_df[valuation_col] = (
            valuation_df[psr_col] * valuation_df['matched_ttm_with_exog']
        )

    company_valuation_results = valuation_df.copy()

else:
    company_valuation_results = pd.DataFrame()

# 8단계: Long Format DB 데이터 생성
forecast_date = pd.Timestamp.now().strftime('%Y-%m-%d')

ttm_combined_data['forecast_date'] = forecast_date
ttm_combined_data['exog_var'] = hs_code if hs_code is not None else None

ttm_data_subset = ttm_combined_data[[
   'date',
   'revenue_with_exog_forecast',
   'revenue_without_exog_forecast',
   'is_forecast',
   'ttm_revenue_with_exog',
   'ttm_revenue_without_exog',
   'forecast_date',
   'exog_var'
]].copy()

if len(company_valuation_results) > 0:
    valuation_data_subset = company_valuation_results[[
       'date',
       'sarima_valuation',
       'prophet_valuation',
       'lstm_valuation',
       'ensemble_valuation',
       'matched_ttm_with_exog',
       'matched_ttm_without_exog'
    ]].copy()
else:
    valuation_data_subset = pd.DataFrame()

ttm_long_list = []

for _, row in ttm_data_subset.iterrows():
   date = row['date']

   items_to_convert = [
       ('revenue_with_exog_forecast', row['revenue_with_exog_forecast']),
       ('revenue_without_exog_forecast', row['revenue_without_exog_forecast']),
       ('is_forecast', 1 if row['is_forecast'] else 0),
       ('ttm_revenue_with_exog', row['ttm_revenue_with_exog']),
       ('ttm_revenue_without_exog', row['ttm_revenue_without_exog']),
       ('forecast_date', row['forecast_date']),
       ('exog_var', row['exog_var'])
   ]

   for item_name, item_value in items_to_convert:
       if pd.notna(item_value):
           ttm_long_list.append({
               'date': date,
               'ticker': tic_name,
               'indicator': item_name,
               'value': item_value
           })

ttm_long_df = pd.DataFrame(ttm_long_list)

valuation_long_list = []

if len(valuation_data_subset) > 0:
    for _, row in valuation_data_subset.iterrows():
       date = row['date']

       items_to_convert = [
           ('sarima_valuation', row['sarima_valuation']),
           ('prophet_valuation', row['prophet_valuation']),
           ('lstm_valuation', row['lstm_valuation']),
           ('ensemble_valuation', row['ensemble_valuation']),
           ('matched_ttm_with_exog', row['matched_ttm_with_exog']),
           ('matched_ttm_without_exog', row['matched_ttm_without_exog']),
           ('forecast_date', forecast_date),
           ('exog_var', hs_code if hs_code is not None else None)
       ]

       for item_name, item_value in items_to_convert:
           if pd.notna(item_value):
               valuation_long_list.append({
                   'date': date,
                   'ticker': tic_name,
                   'indicator': item_name,
                   'value': item_value
               })

valuation_long_df = pd.DataFrame(valuation_long_list)

long_format_for_db = pd.concat([ttm_long_df, valuation_long_df], ignore_index=True)
long_format_for_db = long_format_for_db.sort_values(['date', 'indicator']).reset_index(drop=True)

long_format_for_db['date'] = pd.to_datetime(long_format_for_db['date'])

def convert_value_type(row):
   if row['indicator'] in ['forecast_date', 'exog_var']:
       return str(row['value']) if row['value'] is not None else None
   else:
       try:
           return float(row['value'])
       except:
           return row['value']

long_format_for_db['value'] = long_format_for_db.apply(convert_value_type, axis=1)
long_format_for_db_final = long_format_for_db.copy()

# 9단계: DB 저장
table_name = "Korea_company_valuation_ver2"
save_valuation_to_db(db_info, table_name, long_format_for_db_final)

print(f"기업 가치평가 완료 - {tic_name}")
print(f"총 {len(long_format_for_db_final):,}개 레코드가 DB에 저장되었습니다.")

✅ 'korea_fs_data' 테이블에서 5902708건의 데이터를 가져왔습니다.
해당 기업의 매출 데이터가 없습니다.


23:38:01 - cmdstanpy - INFO - Chain [1] start processing
23:38:01 - cmdstanpy - INFO - Chain [1] done processing


기업 가치평가 완료 - A042700 
총 91개 레코드가 DB에 저장되었습니다.


In [ ]:
company_valuation_results

In [37]:
psr_calculation_df.tail(12)

,date,market_cap,revenue,ttm_revenue,ttm_revenue_shift,psr,predicted_psr,is_forecast
116,2024-09-30,3.671420e+14,7.909873e+10,2.928626e+11,2.671057e+11,1.374519,NaN,False
117,2024-10-31,3.534110e+14,7.909873e+10,2.928626e+11,2.671057e+11,1.323113,NaN,False
118,2024-11-30,3.235620e+14,7.909873e+10,2.928626e+11,2.671057e+11,1.211363,NaN,False
119,2024-12-31,3.175920e+14,7.578827e+10,3.008709e+11,2.811685e+11,1.129543,NaN,False
120,2025-01-31,3.128170e+14,7.578827e+10,3.008709e+11,2.811685e+11,1.112561,NaN,False
121,2025-02-28,3.253530e+14,7.578827e+10,3.008709e+11,2.811685e+11,1.157146,NaN,False
122,2025-03-31,3.421550e+14,7.914050e+10,3.080958e+11,2.928626e+11,1.168312,NaN,False
123,2025-04-30,3.285400e+14,7.914050e+10,3.080958e+11,2.928626e+11,1.121823,NaN,False
124,2025-05-31,3.326840e+14,7.914050e+10,3.080958e+11,2.928626e+11,1.135973,NaN,False
125,2025-06-30,3.539940e+14,7.456632e+10,3.085938e+11,3.008709e+11,1.176564,NaN,False


In [60]:
forecast_df.tail(10)

,date,endog_var,exog_var
76,2023-03-31,9.721347e+07,-49.837932
77,2023-06-30,9.747611e+07,-45.532835
78,2023-09-30,9.351184e+07,-29.089092
79,2023-12-31,9.345371e+07,23.377609
80,2024-03-31,1.234211e+08,75.019108
81,2024-06-30,1.422416e+08,82.526974
82,2024-09-30,1.378387e+08,69.910173
83,2024-12-31,1.370033e+08,50.893537
84,2025-03-31,2.149112e+08,4.568438
85,2025-06-30,2.485939e+08,20.731883


In [31]:
actual_revenue_ttm.tail(10)

,date,revenue,ttm_revenue
76,2023-03-31,6.374537e+10,2.881952e+11
77,2023-06-30,6.000553e+10,2.709972e+11
78,2023-09-30,6.740465e+10,2.616201e+11
79,2023-12-31,6.777994e+10,2.589355e+11
80,2024-03-31,7.191560e+10,2.671057e+11
81,2024-06-30,7.406830e+10,2.811685e+11
82,2024-09-30,7.909873e+10,2.928626e+11
83,2024-12-31,7.578827e+10,3.008709e+11
84,2025-03-31,7.914050e+10,3.080958e+11
85,2025-06-30,7.456632e+10,3.085938e+11


In [61]:
forecast_revenue_df

,date,revenue,predicted_revenue,predicted_revenue_without_exog
0,2025-09-30,NaN,2.511098e+08,2.521870e+08
1,2025-12-31,NaN,2.575981e+08,2.521870e+08
2,2026-03-31,NaN,3.469263e+08,2.521870e+08
3,2026-06-30,NaN,3.844950e+08,2.521870e+08
